# Telco Customer Churn Prediction

## Project Overview
This notebook builds a machine learning model to predict customer churn
for a telecommunications company. We use a real-world dataset of 7,043
customers with 21 features including demographics, services, and account info.

## Goals
- Understand what drives customer churn through exploratory analysis
- Build and compare multiple prediction models
- Identify the most important features for retention decisions

**Dataset:** [IBM Telco Customer Churn (via Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)    
**Author:** Adam Faik

In [1]:
import sys
print(sys.executable)

/usr/local/bin/python3


In [2]:
# --- Imports ---
# We load all our tools at the top so they're available throughout the notebook

import pandas as pd           # pandas handles our data - we nickname it 'pd' to type less
import matplotlib.pyplot as plt  # matplotlib draws charts - 'plt' is the universal nickname
import seaborn as sns          # seaborn makes prettier charts with less code

# This line tells matplotlib to display charts directly inside the notebook
%matplotlib inline

# Print a confirmation so we know everything loaded without errors
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
# --- Load the Dataset ---

# pd.read_csv() reads a CSV file and turns it into a DataFrame
# We store it in a variable called 'df' - short for DataFrame
# 'df' is the universal nickname every data scientist uses
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# This prints a confirmation showing how many rows and columns we loaded
# df.shape returns two numbers: (rows, columns)
print(f"Dataset loaded! Shape: {df.shape}")
print(f"That's {df.shape[0]} customers and {df.shape[1]} features.")

Dataset loaded! Shape: (7043, 21)
That's 7043 customers and 21 features.


In [4]:
# --- First Look at the Data ---

# .head() shows the first 5 rows of the DataFrame
# Like peeking at the top of a spreadsheet
print("=== First 5 rows ===")
df.head()

=== First 5 rows ===


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
# .info() gives us a summary of every column:
# - how many non-null (non-missing) values it has
# - what data type it is (int, float, object=text)
print("=== Dataset Info ===")
df.info()

=== Dataset Info ===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null 

In [6]:
# --- Statistical Summary ---

# .describe() automatically calculates key statistics for every numerical column
# count = how many values, mean = average, std = spread,
# min/max = extremes, 25%/50%/75% = distribution markers
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [7]:
# --- Fix TotalCharges Data Type ---

# First, let's see the problem up close
# We find rows where TotalCharges is just a blank space ' '
# strip() removes whitespace, so '  ' becomes '' (empty string)
problem_rows = df[df['TotalCharges'].str.strip() == '']
print(f"Rows with blank TotalCharges: {len(problem_rows)}")

# Let's see those problem rows
print(problem_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

Rows with blank TotalCharges: 11
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             


In [8]:
# --- Clean TotalCharges ---

# Step 1: Remove the 11 rows where TotalCharges is blank
# We keep only rows where TotalCharges is NOT an empty string after stripping whitespace
df = df[df['TotalCharges'].str.strip() != '']

# Step 2: Convert TotalCharges from text to a decimal number (float)
# pd.to_numeric() does the conversion
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Step 3: Confirm the fix worked
print(f"Rows remaining: {len(df)}")         # Should be 7032 (7043 minus 11)
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")  # Should say float64 now

Rows remaining: 7032
TotalCharges dtype: float64


In [9]:
# --- Drop customerID ---

# .drop() removes a column from the DataFrame
# axis=1 means "drop a column" (axis=0 would mean "drop a row")
# inplace=True means "modify df directly, don't create a copy"
df.drop('customerID', axis=1, inplace=True)

# Confirm it's gone - should now show 20 columns instead of 21
print(f"Columns remaining: {df.shape[1]}")
print(f"Column names: {list(df.columns)}")

Columns remaining: 20
Column names: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [10]:
# --- Convert Churn to Binary (0/1) ---

# .map() replaces every value in a column using a dictionary
# 'No' gets replaced with 0, 'Yes' gets replaced with 1
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

# Confirm the conversion worked
print(f"Churn value counts:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {df['Churn'].mean():.1%}")  # .mean() on 0/1 gives us the % of 1s

Churn value counts:
Churn
0    5163
1    1869
Name: count, dtype: int64

Churn rate: 26.6%


In [11]:
# --- Final Data Quality Check ---

# .isnull().sum() counts missing values in every column
# A column with 0 means no missing values - exactly what we want
print("=== Missing values per column ===")
print(df.isnull().sum())

# Confirm final shape
print(f"\n=== Final dataset shape ===")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")

# Quick peek at the cleaned data
print(f"\n=== First 3 rows of clean data ===")
df.head(3)

=== Missing values per column ===
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

=== Final dataset shape ===
7032 rows, 20 columns

=== First 3 rows of clean data ===


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
